# MGT 599 Capstone - Group 4
## Descriptive Analytics

April 2026

This notebook goes through both datasets and looks at the data in detail - shapes, missing values, class distributions, revenue numbers, text lengths, word frequencies, and correlations.

## Setup

In [ ]:
import sys
import re
import warnings
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
sys.path.insert(0, str(PROJECT_ROOT))

from common_utils import (
    CLEANED_PATH, ensure_folders,
    load_task_dataframe, guess_target_column,
    guess_text_columns, safe_text, set_chart_style
)

set_chart_style()
ensure_folders()
print('ready')

## 1. Dataset overview

In [ ]:
task1 = load_task_dataframe('task1')
task2 = load_task_dataframe('task2')

t1_target = guess_target_column(task1, 'task1')
t2_target = guess_target_column(task2, 'task2')
t1_text_cols = guess_text_columns(task1)
t2_text_cols = guess_text_columns(task2)

print('task 1')
print(f'  rows: {task1.shape[0]:,}  cols: {task1.shape[1]}')
print(f'  target: {t1_target}  classes: {task1[t1_target].nunique()}')
print()
print('task 2')
print(f'  rows: {task2.shape[0]:,}  cols: {task2.shape[1]}')
print(f'  target: {t2_target}  classes: {task2[t2_target].nunique()}')

In [ ]:
print('task 1 column types:')
display(task1.dtypes.rename('dtype').to_frame())
print('\ntask 2 column types:')
display(task2.dtypes.rename('dtype').to_frame())

In [ ]:
print('task 1 numeric stats:')
display(task1.describe().T.round(2))

In [ ]:
print('task 2 numeric stats:')
display(task2.describe().T.round(2))

## 2. Missing values and data quality

In [ ]:
def missing_report(df, name):
    report = pd.DataFrame({
        'column':          df.columns,
        'missing count':   [df[c].isna().sum() for c in df.columns],
        'missing %':       [round(df[c].isna().mean() * 100, 2) for c in df.columns],
        'unique values':   [df[c].nunique() for c in df.columns],
        'dtype':           [str(df[c].dtype) for c in df.columns],
    }).sort_values('missing count', ascending=False).reset_index(drop=True)
    print(f'{name}:')
    display(report)
    print(f'  duplicate rows: {df.duplicated().sum()}')

missing_report(task1, 'task 1')
missing_report(task2, 'task 2')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (df, name) in zip(axes, [(task1, 'task 1'), (task2, 'task 2')]):
    pct = (df.isna().mean() * 100).values.reshape(1, -1)
    sns.heatmap(
        pct, ax=ax,
        xticklabels=df.columns,
        yticklabels=['% missing'],
        cmap='YlOrRd', vmin=0, vmax=100,
        annot=True, fmt='.1f', linewidths=0.5
    )
    ax.set_title(f'{name} - missing % by column')
    ax.tick_params(axis='x', rotation=40)

plt.tight_layout()
plt.show()

## 3. Revenue analysis (task 1 numeric columns)

In [ ]:
if 'Revenue' in task1.columns:
    rev = task1['Revenue'].dropna()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].hist(rev, bins=80, color='#2E75B6', edgecolor='white')
    axes[0].set_title('revenue distribution')
    axes[0].set_xlabel('revenue')
    axes[0].axvline(rev.median(), color='red', linestyle='--',
                    label=f'median={rev.median():,.0f}')
    axes[0].legend()

    axes[1].hist(rev[rev > 0], bins=80, color='#4BACC6', edgecolor='white', log=True)
    axes[1].set_title('revenue - log scale')
    axes[1].set_xlabel('revenue')

    axes[2].boxplot(rev, patch_artist=True,
                    boxprops=dict(facecolor='#2E75B6', alpha=0.7))
    axes[2].set_title('revenue box plot')

    plt.tight_layout()
    plt.show()

    print(f'min: {rev.min():,.0f}  max: {rev.max():,.0f}')
    print(f'median: {rev.median():,.0f}  mean: {rev.mean():,.0f}')
    print(f'zeros: {(rev==0).sum():,}  negatives: {(rev<0).sum():,}')

In [ ]:
if 'revenue_share' in task1.columns:
    rs = task1['revenue_share'].dropna()
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(rs, bins=50, color='#9B59B6', edgecolor='white')
    axes[0].set_title('revenue share distribution')
    axes[0].axvline(rs.median(), color='red', linestyle='--',
                    label=f'median={rs.median():.2f}')
    axes[0].legend()

    if 'is_largest_share_segment' in task1.columns:
        vc = task1['is_largest_share_segment'].value_counts()
        axes[1].pie(vc.values, labels=[f'{k} ({v:,})' for k,v in vc.items()],
                    autopct='%1.1f%%', colors=['#2E75B6','#C0504D'], startangle=90)
        axes[1].set_title('is largest share segment')

    plt.tight_layout()
    plt.show()

## 4. Class distribution

In [ ]:
t1_vc = task1[t1_target].value_counts()

print('task 1 -', t1_target)
print(f'  total classes: {len(t1_vc)}')
print(f'  top class: {t1_vc.index[0]}  ({t1_vc.iloc[0]:,} records)')
print(f'  smallest: {t1_vc.index[-1]}  ({t1_vc.iloc[-1]:,} records)')
print(f'  classes with <10 rows: {(t1_vc<10).sum()}')
print(f'  top 3 classes cover: {t1_vc.head(3).sum()/len(task1)*100:.1f}% of data')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 14))

t1_vc.head(20).sort_values().plot(kind='barh', ax=axes[0], color='#2E75B6')
axes[0].set_title('top 20 industry classes - task 1')
axes[0].set_xlabel('number of records')
for i, v in enumerate(t1_vc.head(20).sort_values().values):
    axes[0].text(v + t1_vc.max()*0.005, i, f'{v:,}', va='center', fontsize=8)

t1_vc.tail(20).sort_values(ascending=False).plot(kind='barh', ax=axes[1], color='#C0504D')
axes[1].set_title('rarest 20 industry classes - task 1')
axes[1].set_xlabel('number of records')

plt.tight_layout()
plt.show()

In [ ]:
buckets = pd.cut(t1_vc, bins=[0,10,50,100,500,1000,float('inf')],
                 labels=['1-10','11-50','51-100','101-500','501-1000','1000+'])
bc = buckets.value_counts().sort_index()

plt.figure(figsize=(10, 5))
bc.plot(kind='bar', color='#4472C4', edgecolor='white')
plt.title('task 1 - classes by record count bucket')
plt.xlabel('records per class')
plt.ylabel('number of classes')
plt.xticks(rotation=0)
for i, v in enumerate(bc.values):
    plt.text(i, v+0.3, str(v), ha='center', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
t2_vc = task2[t2_target].value_counts()

print('task 2 -', t2_target)
print(f'  total classes: {len(t2_vc)}')
print(f'  top class: {t2_vc.index[0]}  ({t2_vc.iloc[0]:,} records)')
print(f'  smallest: {t2_vc.index[-1]}  ({t2_vc.iloc[-1]:,} records)')
print(f'  classes with <10 rows: {(t2_vc<10).sum()}')
print(f'  top 3 classes cover: {t2_vc.head(3).sum()/len(task2)*100:.1f}% of data')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 18))

t2_vc.head(25).sort_values().plot(kind='barh', ax=axes[0], color='#4BACC6')
axes[0].set_title('top 25 subindustry classes - task 2')
axes[0].set_xlabel('number of records')

t2_vc.tail(25).sort_values(ascending=False).plot(kind='barh', ax=axes[1], color='#9B59B6')
axes[1].set_title('rarest 25 subindustry classes - task 2')
axes[1].set_xlabel('number of records')

plt.tight_layout()
plt.show()

## 5. Text column analysis

In [ ]:
def text_stats(df, col, label):
    txt = safe_text(df[col])
    char_len = txt.apply(len)
    word_cnt = txt.apply(lambda x: len(x.split()))
    empty = (txt.str.strip() == '').sum()
    short = (word_cnt < 5).sum()
    return pd.DataFrame([{
        'dataset':    label,
        'column':     col,
        'empty rows': empty,
        'empty %':    round(empty/len(df)*100, 1),
        'short (<5w)':short,
        'short %':    round(short/len(df)*100, 1),
        'avg chars':  round(char_len.mean(), 0),
        'med chars':  round(char_len.median(), 0),
        'max chars':  char_len.max(),
        'avg words':  round(word_cnt.mean(), 1),
        'med words':  round(word_cnt.median(), 1),
    }])

text_summary = pd.concat(
    [text_stats(task1, col, 'task 1') for col in t1_text_cols] +
    [text_stats(task2, col, 'task 2') for col in t2_text_cols],
    ignore_index=True
)
display(text_summary)

In [ ]:
all_pairs = [(task1, c, 'task 1') for c in t1_text_cols] + \
            [(task2, c, 'task 2') for c in t2_text_cols]

n = len(all_pairs)
if n > 0:
    fig, axes = plt.subplots(n, 2, figsize=(16, 5*n), squeeze=False)
    for i, (df, col, label) in enumerate(all_pairs):
        txt = safe_text(df[col])
        chars = txt.apply(len)
        words = txt.apply(lambda x: len(x.split()))

        axes[i][0].hist(chars, bins=60, color='#2E75B6', edgecolor='white', alpha=0.85)
        axes[i][0].axvline(chars.median(), color='red', linestyle='--',
                            label=f'median={chars.median():.0f}')
        axes[i][0].set_title(f'{label} / {col} - char length')
        axes[i][0].set_xlabel('characters')
        axes[i][0].legend()

        axes[i][1].hist(words, bins=60, color='#4BACC6', edgecolor='white', alpha=0.85)
        axes[i][1].axvline(words.median(), color='red', linestyle='--',
                            label=f'median={words.median():.0f}')
        axes[i][1].set_title(f'{label} / {col} - word count')
        axes[i][1].set_xlabel('words')
        axes[i][1].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
STOPWORDS = {
    'the','and','of','a','in','to','is','that','for','with','its',
    'are','as','by','an','it','or','has','on','from','be','this','at',
    'which','was','have','also','their','more','been','than','but','all',
    'not','through','into','other','these','such','over','about','they',
    'we','our','can','will','well','who','based'
}

if t1_text_cols:
    all_words = ' '.join(safe_text(task1[t1_text_cols[0]]).str.lower()).split()
    filtered = [w for w in all_words
                if w not in STOPWORDS and len(w) > 2 and re.match(r'^[a-z]+$', w)]
    top30 = pd.DataFrame(Counter(filtered).most_common(30), columns=['word','count'])

    plt.figure(figsize=(14, 6))
    plt.barh(top30['word'][::-1], top30['count'][::-1], color='#2E75B6')
    plt.title(f'top 30 words in task 1 - {t1_text_cols[0]}')
    plt.xlabel('frequency')
    plt.tight_layout()
    plt.show()

In [ ]:
if t2_text_cols:
    all_words2 = ' '.join(safe_text(task2[t2_text_cols[0]]).str.lower()).split()
    filtered2 = [w for w in all_words2
                 if w not in STOPWORDS and len(w) > 2 and re.match(r'^[a-z]+$', w)]
    top30_2 = pd.DataFrame(Counter(filtered2).most_common(30), columns=['word','count'])

    plt.figure(figsize=(14, 6))
    plt.barh(top30_2['word'][::-1], top30_2['count'][::-1], color='#4BACC6')
    plt.title(f'top 30 words in task 2 - {t2_text_cols[0]}')
    plt.xlabel('frequency')
    plt.tight_layout()
    plt.show()

## 6. Cross-analysis

In [ ]:
if 'Revenue' in task1.columns:
    rev_by_ind = (
        task1.groupby(t1_target)['Revenue']
        .agg(['median','mean','count'])
        .rename(columns={'median':'median revenue','mean':'mean revenue','count':'records'})
        .sort_values('median revenue', ascending=False)
        .reset_index()
    )

    top15 = rev_by_ind.head(15)
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    top15.sort_values('median revenue').plot(
        kind='barh', x=t1_target, y='median revenue', ax=axes[0], color='#2E75B6', legend=False
    )
    axes[0].set_title('top 15 industries by median revenue')

    top15.sort_values('records').plot(
        kind='barh', x=t1_target, y='records', ax=axes[1], color='#4BACC6', legend=False
    )
    axes[1].set_title('top 15 industries by record count')

    plt.tight_layout()
    plt.show()
    display(rev_by_ind.head(10).round(0))

In [ ]:
if 'CompanyId' in task1.columns:
    segs = task1.groupby('CompanyId').size()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(segs, bins=40, color='#9B59B6', edgecolor='white')
    axes[0].set_title('segments per company - task 1')
    axes[0].set_xlabel('number of segments')
    axes[0].axvline(segs.median(), color='red', linestyle='--',
                    label=f'median={segs.median():.0f}')
    axes[0].legend()

    vc = segs.value_counts().sort_index().head(15)
    axes[1].bar(vc.index, vc.values, color='#9B59B6', edgecolor='white')
    axes[1].set_title('companies by segment count')
    axes[1].set_xlabel('segments')

    plt.tight_layout()
    plt.show()

    print(f'unique companies: {segs.shape[0]:,}')
    print(f'avg segments per company: {segs.mean():.1f}')
    print(f'max segments: {segs.max()}')
    print(f'single-segment companies: {(segs==1).sum():,}')

In [ ]:
if 'revenue_share' in task1.columns:
    top_ind = task1[t1_target].value_counts().head(10).index.tolist()
    subset = task1[task1[t1_target].isin(top_ind)]
    order = subset.groupby(t1_target)['revenue_share'].median().sort_values(ascending=False).index

    plt.figure(figsize=(16, 7))
    sns.boxplot(data=subset, x=t1_target, y='revenue_share', order=order, palette='Blues')
    plt.title('revenue share by industry - top 10 industries')
    plt.xticks(rotation=40, ha='right')
    plt.tight_layout()
    plt.show()

## 7. Correlation matrix

In [ ]:
num_cols = task1.select_dtypes(include='number').columns.tolist()

if len(num_cols) >= 2:
    corr = task1[num_cols].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))

    plt.figure(figsize=(10, 7))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
                cmap='coolwarm', center=0, linewidths=0.5, vmin=-1, vmax=1)
    plt.title('task 1 - correlation matrix')
    plt.tight_layout()
    plt.show()

    display(corr.round(3))

## 8. Full descriptive stats

In [ ]:
print('task 1:')
display(task1.describe(include='all').T.rename(columns={'50%':'median'}).round(2))

In [ ]:
print('task 2:')
display(task2.describe(include='all').T.rename(columns={'50%':'median'}).round(2))

## What we found

Class imbalance is the main issue. A few industry categories have thousands of rows but a lot of the smaller ones have fewer than 10. If we train a model without handling this it will just overfit on the big categories.

Revenue is heavily right-skewed. A small number of very large companies pull the mean way up. The median is more representative. Log transformation might help if we use revenue as a feature.

Some description fields are empty or very short. The model needs text to work with so those rows are flagged.

The data is at the segment level so one company can have many rows, one per business segment.

The word frequency charts show that the descriptions do have industry-specific vocabulary which is encouraging for TF-IDF-based features.